## Word Embeddings with Word2Vec

---

## Part 1: Setup and Data Loading

In [14]:
!pip install gensim datasets scikit-learn matplotlib -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import re
import random

from gensim.models import Word2Vec
import gensim.downloader as api
from sklearn.manifold import TSNE

random.seed(103)
np.random.seed(103)

print("✅ Libraries loaded!")

✅ Libraries loaded!


In [15]:
from datasets import load_dataset

# Need lots of text for good embeddings
dataset = load_dataset("stanfordnlp/imdb", split="train")
texts = dataset['text']

print(f"✅ Loaded {len(texts):,} documents")

✅ Loaded 25,000 documents


## Part 2: Prepare Text for Word2Vec

In [16]:
def preprocess_for_word2vec(text):
    """Clean and tokenize text for Word2Vec."""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if len(t) > 2]
    return tokens

print("Preprocessing texts...")
corpus = [preprocess_for_word2vec(text) for text in texts]
corpus = [doc for doc in corpus if len(doc) > 5]

print(f"✅ Prepared {len(corpus):,} documents")

Preprocessing texts...
✅ Prepared 25,000 documents


## Part 3: Train Word2Vec Model

In [17]:
print("🔧 Training Word2Vec model (1-2 minutes)...")

model = Word2Vec(
    sentences=corpus,
    vector_size=100,
    window=5,
    min_count=10,
    workers=4,
    sg=1,
    epochs=15,
    seed=103
)

print(f"✅ Model trained!")
print(f"   Vocabulary: {len(model.wv):,} words")

🔧 Training Word2Vec model (1-2 minutes)...
✅ Model trained!
   Vocabulary: 19,656 words


## Part 4: Explore Word Similarities

In [18]:
test_words = ["good", "bad", "movie", "actor", "director"]

print("📊 WORD SIMILARITIES")
print("=" * 60)

for word in test_words:
    if word in model.wv:
        similar = model.wv.most_similar(word, topn=5)
        print(f"\n'{word}' is similar to:")
        for similar_word, score in similar:
            print(f"   {similar_word:15} {score:.3f}")

📊 WORD SIMILARITIES

'good' is similar to:
   great           0.754
   decent          0.748
   excellent       0.694
   nice            0.687
   solid           0.673

'bad' is similar to:
   terrible        0.718
   awful           0.708
   horrible        0.682
   good            0.670
   lousy           0.658

'movie' is similar to:
   film            0.871
   flick           0.683
   movies          0.632
   this            0.630
   programme       0.605

'actor' is similar to:
   actress         0.719
   role            0.688
   dependable      0.665
   performance     0.648
   actors          0.635

'director' is similar to:
   writer          0.811
   filmmaker       0.769
   screenwriter    0.758
   cinematographer 0.727
   producer        0.717


## Part 5: Document Failures

In [19]:
# Antonym problem: antonyms often appear similar!
print("🔍 ANTONYM TEST")
print("-" * 40)

antonym_pairs = [
    ("good", "bad"),
    ("love", "hate"),
    ("best", "worst"),
]

for word1, word2 in antonym_pairs:
    if word1 in model.wv and word2 in model.wv:
        sim = model.wv.similarity(word1, word2)
        problem = "⚠️ TOO SIMILAR!" if sim > 0.3 else "✓ OK"
        print(f"{word1:10} ↔ {word2:10} : {sim:.3f} {problem}")

🔍 ANTONYM TEST
----------------------------------------
good       ↔ bad        : 0.670 ⚠️ TOO SIMILAR!
love       ↔ hate       : 0.657 ⚠️ TOO SIMILAR!
best       ↔ worst      : 0.723 ⚠️ TOO SIMILAR!


In [ ]:
# FAILURE HUNT: Find at least 1 case where embeddings give wrong results
# Try words I expect to be similar but aren't, or different but are similar

# YOUR CODE HERE:
pairs = [
    ("happy", "sad"),
    ("big", "small"),
    ("many", "few")
]

for word1, word2 in pairs:
    if word1 in model.wv and word2 in model.wv:
        sim = model.wv.similarity(word1, word2)
        problem = "⚠️ TOO SIMILAR!" if sim > 0.3 else "✓ OK"
        print(f"{word1:10} ↔ {word2:10} : {sim:.3f} {problem}")

happy      ↔ sad        : 0.538 ⚠️ TOO SIMILAR!
big        ↔ small      : 0.583 ⚠️ TOO SIMILAR!
many       ↔ few        : 0.664 ⚠️ TOO SIMILAR!


## Part 6: Business Analogies

In [21]:
def test_analogy(word1, word2, word3, model):
    """Test: word1 - word2 + word3 = ?"""
    try:
        result = model.wv.most_similar(
            positive=[word1, word3],
            negative=[word2],
            topn=3
        )
        return result
    except KeyError as e:
        return f"Word not in vocabulary: {e}"

# Test some analogies
analogies = [
    ("good", "better", "bad"),  # bad + (better - good) = worse?
]

for word1, word2, word3 in analogies:
    result = test_analogy(word1, word2, word3, model)
    print(f"{word3} - {word2} + {word1} = ?")
    if isinstance(result, list):
        for word, score in result:
            print(f"   → {word} ({score:.3f})")

bad - better + good = ?
   → cool (0.523)
   → terrible (0.519)
   → poor (0.504)


In [ ]:
# ANALOGIES: Create at least 3 business-relevant analogies
my_analogies = [
    ("terrible", "awful", "good"),
    ("director", "film", "writer"),
    ("actor", "bad", "director")
]

for word1, word2, word3 in my_analogies:
    result = test_analogy(word1, word2, word3, model)
    print(f"\n{word3} - {word2} + {word1} = ?")
    if isinstance(result, list):
        for word, score in result:
            print(f"   → {word} ({score:.3f})")


good - awful + terrible = ?
   → decent (0.690)
   → great (0.678)
   → excellent (0.615)

writer - film + director = ?
   → screenwriter (0.658)
   → producer (0.636)
   → writers (0.574)

director - bad + actor = ?
   → writer (0.624)
   → filmmaker (0.614)
   → aja (0.595)


## Part 7: Compare to Pre-trained

In [23]:
print("📥 Loading pre-trained GloVe...")
glove = api.load("glove-twitter-50")
print(f"✅ Loaded {len(glove):,} words")

# Compare
compare_word = "good"
print(f"\nSimilar to '{compare_word}':")
print(f"Your model:  {[w for w, _ in model.wv.most_similar(compare_word, topn=5)]}")
print(f"Pre-trained: {[w for w, _ in glove.most_similar(compare_word, topn=5)]}")

📥 Loading pre-trained GloVe...
✅ Loaded 1,193,514 words

Similar to 'good':
Your model:  ['great', 'decent', 'excellent', 'nice', 'solid']
Pre-trained: ['well', 'great', 'too', 'nice', 'better']
